# 01 - Download GA sample dataset (quarterly Parquet)
Source: `bigquery-public-data.google_analytics_sample.ga_sessions_*`

**Billing is enabled on this project, so overspending is possible.** Run the cells in order. The download cell will not run until you set an approved cap after reading the dry-run result.

In [ ]:
%pip install -q google-cloud-bigquery pyarrow db-dtypes duckdb

## Step 1 - Configure
Set the environment variable `GCP_PROJECT` before starting Jupyter, or replace the placeholder below (do not commit your project ID).

In [ ]:
import os, re, json, hashlib, pathlib, datetime as dt
from collections import defaultdict
from google.cloud import bigquery

PROJECT = os.environ.get("GCP_PROJECT")
client = bigquery.Client(project=PROJECT)
RAW = pathlib.Path("data/raw/ga_sessions"); RAW.mkdir(parents=True, exist_ok=True)
SRC = "`bigquery-public-data.google_analytics_sample.ga_sessions_*`"
print("Using project:", client.project)

In [ ]:
ds = bigquery.DatasetReference("bigquery-public-data", "google_analytics_sample")
names = [t.table_id for t in client.list_tables(ds)]
pat = re.compile(r"ga_sessions_\d{8}")
days = sorted(n.split("_")[-1] for n in names if pat.fullmatch(n))
other = [n for n in names if not pat.fullmatch(n)]
print(len(days), "daily tables:", days[0], "->", days[-1])
print("Other tables:", other)

d0, d1 = (dt.datetime.strptime(x, "%Y%m%d").date() for x in (days[0], days[-1]))
full = {(d0 + dt.timedelta(i)).strftime("%Y%m%d") for i in range((d1 - d0).days + 1)}
print("Missing days:", sorted(full - set(days)))

(RAW / "_manifest").mkdir(exist_ok=True)
(RAW / "_manifest" / "tables.json").write_text(json.dumps({"days": days, "other": other}))

qs = defaultdict(list)
for d in days:
    qs[f"{d[:4]}Q{(int(d[4:6]) - 1)//3 + 1}"].append(d)
quarters = {q: (v[0], v[-1]) for q, v in sorted(qs.items())}
for q, (a, b) in quarters.items():
    print(q, a, b, f"{len(qs[q])} days")

## Step 3 - Dry run (free, nothing is executed)
Shows how many bytes the full extraction would process. The free allowance is 1 TB per month; anything above that is billed to your company account.

In [ ]:
def dry(sql, params=None):
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False, query_parameters=params or [])
    return client.query(sql, job_config=cfg).total_bytes_processed

total = dry(f"SELECT * FROM {SRC}")
print(f"Full extraction would process {total/1024**3:.2f} GiB ({total/1024**4*100:.2f}% of 1 TiB)")

SQL = f"SELECT * FROM {SRC} WHERE _TABLE_SUFFIX BETWEEN @a AND @b"
def qparams(a, b):
    return [bigquery.ScalarQueryParameter("a", "STRING", a), bigquery.ScalarQueryParameter("b", "STRING", b)]
per_q = {q: dry(SQL, qparams(a, b)) for q, (a, b) in quarters.items()}
for q, n in per_q.items():
    print(q, f"{n/1024**3:.2f} GiB")

## Step 4 - Download (requires your approval)
Set `CAP_GIB` to a number you accept for **each** quarter (for example slightly above the largest value printed above). Leave `ONLY` as a test quarter first, then set it to `None` for all quarters. Already-downloaded quarters are skipped, so it is safe to re-run.

In [ ]:
import pyarrow.parquet as pq

CAP_GIB = 2          # <-- set after reading Step 3, e.g. 3
ONLY = None       # <-- test with one quarter first; None = all

assert CAP_GIB is not None, "Set CAP_GIB first (you approve the spend limit)."
cap = int(CAP_GIB * 1024**3)

for q, (a, b) in quarters.items():
    if ONLY and q not in ONLY:
        continue
    dest = RAW / f"{q}.parquet"
    if dest.exists():
        print(q, "already downloaded, skipping"); continue
    if per_q[q] > cap:
        print(q, f"estimate {per_q[q]/1024**3:.2f} GiB exceeds cap, skipped"); continue
    job = client.query(SQL, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=cap, query_parameters=qparams(a, b)))
    table = job.result().to_arrow(create_bqstorage_client=False)
    tmp = dest.with_suffix(".tmp")
    pq.write_table(table, tmp); tmp.replace(dest)
    (RAW / f"{q}.manifest.json").write_text(json.dumps({
        "quarter": q, "range": [a, b], "rows": table.num_rows,
        "sha256": hashlib.sha256(dest.read_bytes()).hexdigest(),
        "bytes_processed": job.total_bytes_processed,
        "downloaded_on": dt.date.today().isoformat(),
        "source": "bigquery-public-data.google_analytics_sample.ga_sessions_*"}))
    print(q, "saved", table.num_rows, "rows")

## Step 5 - Validate

In [ ]:
import json, hashlib, pathlib, duckdb

RAW = pathlib.Path(r"data/raw/ga_sessions")

for m in sorted(RAW.glob("*.manifest.json")):
    info = json.loads(m.read_text())
    f = RAW / f"{info['quarter']}.parquet"
    ok = hashlib.sha256(f.read_bytes()).hexdigest() == info["sha256"]
    print(info["quarter"], "checksum ok" if ok else "CHECKSUM MISMATCH", info["rows"], "rows")

glob = str(RAW / "*.parquet").replace("\\", "/")
print(duckdb.sql(f"""
  SELECT substr(date,1,4) AS year, quarter(strptime(date,'%Y%m%d')) AS q,
         count(*) AS sessions,
         count(*) FILTER (WHERE totals.transactions IS NOT NULL) AS with_transactions,
         min(date) AS first_day, max(date) AS last_day
  FROM '{glob}' GROUP BY 1,2 ORDER BY 1,2""").df())

print(duckdb.sql(f"""
  SELECT count(*) AS total, count(DISTINCT (fullVisitorId, visitId)) AS distinct_sessions
  FROM '{glob}'""").df())

print(duckdb.sql(f"DESCRIBE SELECT * FROM '{glob}'").df().head(20))